In [1]:
import pickle
import numpy as np
import pandas as pd
import os
import glob
import re

# === Load optimization inputs ===
with open("optimization_inputs.pkl", "rb") as f:
    optimization_inputs = pickle.load(f)

# === Step 1: Read claim counts from original files (safely by service index)
services_dir = "services"  # Adjust if needed
service_files = sorted(glob.glob(os.path.join(services_dir, "services_*.xlsx")))

# Initialize claim count list
claim_counts = [0] * len(optimization_inputs)  # length 11

# Use regex to ensure correct alignment by service number
for file in service_files:
    match = re.match(r".*services_(\d+)_\d+\.xlsx", os.path.basename(file))
    if match:
        service_idx = int(match.group(1))  # 1-based index
        count = pd.read_excel(file).shape[0]
        claim_counts[service_idx - 1] = count
    else:
        raise ValueError(f"Invalid filename format: {file}")

print(" Matched claim counts:", claim_counts)



 Matched claim counts: [2, 7, 18, 19, 41, 75, 128, 962, 36, 143, 146]


In [2]:
# === Step 2: Simulate and export synthetic dataset ===
output_dir = "simulated_dataset_1"
os.makedirs(output_dir, exist_ok=True)

for i, service in enumerate(optimization_inputs):
    n = claim_counts[i]
    c_ij = service["c_ij"]
    p_ij = service["p_ij"]

    # Generate claims
    claims = np.random.choice(c_ij, size=n, p=p_ij)

    df = pd.DataFrame({
        "PersonID": range(1, n + 1),
        "ClaimedAmount": claims
    })

    filename = f"services_{i+1}_{int(service['limit'])}.xlsx"
    df.to_excel(os.path.join(output_dir, filename), index=False)

print(f"\n Synthetic dataset exported to: {output_dir}")




 Synthetic dataset exported to: simulated_dataset_1


In [3]:
import pickle
import numpy as np
import pandas as pd
import os
import glob
import re

# === Load bin distributions from pickle ===
with open("optimization_inputs.pkl", "rb") as f:
    optimization_inputs = pickle.load(f)

# === Load real claim counts from original xlsx filenames ===
services_dir = "services"  # <- Adjust if needed
xlsx_files = sorted(glob.glob(os.path.join(services_dir, "services_*.xlsx")))

# Extract row counts per service, correctly matched to index
claim_counts = [0] * len(optimization_inputs)

for file in xlsx_files:
    match = re.match(r".*services_(\d+)_\d+\.xlsx", os.path.basename(file))
    if match:
        service_idx = int(match.group(1))  # 1-based
        df = pd.read_excel(file)
        claim_counts[service_idx - 1] = df.shape[0]
    else:
        raise ValueError(f"Filename format invalid: {file}")

# === Limits ===
original_limits = [s["limit"] for s in optimization_inputs]
adjusted_values = [
    420000,
    1436108,
    630000,
    210000,
    31500,
    313856,
    103535,
    98710,
    105000,
    483275,
    58013,
]  # improved solver: adaptive ε, weighted objective, λ=0.01

# === Reimbursement simulator ===
def simulate_reimbursement_once():
    original_total = 0
    adjusted_total = 0

    for i in range(len(optimization_inputs)):
        c_ij = optimization_inputs[i]["c_ij"]
        p_ij = optimization_inputs[i]["p_ij"]
        n_claims = claim_counts[i]

        claims = np.random.choice(c_ij, size=n_claims, p=p_ij)
        original_total += np.clip(claims, None, original_limits[i]).sum()
        adjusted_total += np.clip(claims, None, adjusted_values[i]).sum()

    return int(original_total), int(adjusted_total)

# === Run simulations and collect differences ===
differences = []

print("===== MONTE CARLO REIMBURSEMENT SIMULATION (100 RUNS) =====\n")

for i in range(1, 101):
    orig, adj = simulate_reimbursement_once()
    diff = adj - orig
    differences.append(diff)

    print(f"[#{i:03}] Original Total Reimbursed: {orig:,} AMD")
    print(f"     Adjusted Total Reimbursed: {adj:,} AMD")
    print(f"     Change in Reimbursement: {diff:+,} AMD\n")

# === Final Statistics ===
differences = np.array(differences)
mean_diff = np.mean(differences)
var_diff = np.var(differences)

print("===== SUMMARY OF DIFFERENCES ACROSS 100 SIMULATIONS =====")
print(f"Mean Change in Reimbursement: {mean_diff:,.2f} AMD")
print(f"Standard Deviation of Change: {np.sqrt(var_diff):,.2f} AMD")



===== MONTE CARLO REIMBURSEMENT SIMULATION (100 RUNS) =====

[#001] Original Total Reimbursed: 102,812,857 AMD
     Adjusted Total Reimbursed: 102,821,296 AMD
     Change in Reimbursement: +8,439 AMD

[#002] Original Total Reimbursed: 106,049,285 AMD
     Adjusted Total Reimbursed: 105,976,556 AMD
     Change in Reimbursement: -72,729 AMD

[#003] Original Total Reimbursed: 102,767,857 AMD
     Adjusted Total Reimbursed: 102,529,946 AMD
     Change in Reimbursement: -237,911 AMD

[#004] Original Total Reimbursed: 100,045,714 AMD
     Adjusted Total Reimbursed: 99,876,224 AMD
     Change in Reimbursement: -169,490 AMD

[#005] Original Total Reimbursed: 104,645,714 AMD
     Adjusted Total Reimbursed: 104,397,305 AMD
     Change in Reimbursement: -248,409 AMD

[#006] Original Total Reimbursed: 107,587,142 AMD
     Adjusted Total Reimbursed: 107,339,146 AMD
     Change in Reimbursement: -247,996 AMD

[#007] Original Total Reimbursed: 106,452,142 AMD
     Adjusted Total Reimbursed: 106,260,8